Our ability to decode spatial efficiency might be a function of head direction encoding in the retrosplenial cortex. If this is true, any difference in behavioural sampling of head direction might permit a decoder to have above chance in accuracy. Thus checking that the highest coefficients do not have high rayleighs for head direction can support our claim that they are not head direction cells but are actually subgoal helper cells.

From robin about doing it properly - Usually using a measure called KL divergence (Kullback-Liebler). Obvs read the Wikipedia entry on it but basically it's a measure of how much 2 distributions differ. In the case of hd cells you feed in two distributions, a uniform circular one (the expected dist under the null hypothesis) and your observed dist i.e. the polar plot from your cell. If I remember correctly, both distributions are normed to a max of 1. Ideally you'd generate a distribution of the KL score from a load of shuffled data to get your 95th or 99th prctile. In my experience anything with a score over about 0.15 is a hd cell

Can I use rayleigh vector? With some cut off
New

rhayman
  5:27 PM
Yeah could do. Again it's a case of figuring out what a statistically valid cut off is. Time to get a shuffling :man_dancing: (edited) 

Laurence Freeman
  5:29 PM
Ok thanks mate

rhayman
  5:38 PM
Code for KL div is here, the two functions only need numpy and you should be able to feed in just the binned directional 1D ratemap into the first function to get the KL score (famous last words)...
https://github.com/rhayman/ephysiopy/blob/master/ephysiopy%2Fcommon%2Ffieldcalcs.py#L622-L725

In [1]:
# Approach 1: Use an industry defined definition of head direction cells
# I could have a percentage of cells. I.e what are the percentage of cells out of the top coefficients that are similar and have high rayleigh score

# Logic for classifying head direction cells 

In [1]:
from loguru import logger

from behave_analysis.utils.rayleigh.load_rayleigh import extract_rayleigh_path, load_rayleigh_data
from behave_analysis.utils.rayleigh.manipulate_rayleigh_df import extract_compartment_values
from behave_analysis.analyze.classification.head_direction import angle_similarity, rayleigh_threshold, check_both_compartments_significant
from behave_analysis.process.session import get_experiment


In [2]:
# User defined constants
#RAYLEIGH_THRESHOLD = 0.5
SIMILAR_ANGLE_THRESHOLD = 0.8

In [7]:
angle_keys = ['hdir_Rayleigh.arrow', 'hsa_Rayleigh.arrow', 'h_postflipbar_a_Rayleigh.arrow', 'h_preflipbar_a_Rayleigh.arrow']

In [3]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept
from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept
from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept
from behave_analysis.database.Experiments.JAL006_ex import JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr
from behave_analysis.database.Experiments.JAL007_ex import (
    JAL7_sesh8_9apr,
    JAL7_sesh9_16apr,
    JAL7_flip5_22mar,
    JAL7_flip2_12mar,
    JAL7_23apr,
    JAL7_30apr,
)
from behave_analysis.database.Experiments.JAL008_ex import (
    JAL8_flip1_25apr,
    JAL8_flip2_29apr,
    JAL8_tiny_3may,
    JAL8_flip4_10may,
    JAL8_14may,
    JAL8_21may,
    JAL8_flip3_7may,
)

# The values must be in order of data for the code to work
mice_groups = {
    "JAL6": ["JAL6_flip3_18mar", "JAL6_flip4_21mar", "JAL6_flip5_25mar", "JAL6_28mar"],
    "JAL3": ["JAL3_25aug", "JAL3_1sept", "JAL3_4sept", "JAL3_7sept"],
    "JAL7": ["JAL7_flip2_12mar", "JAL7_flip5_22mar", "JAL7_sesh8_9apr", "JAL7_sesh9_16apr", "JAL7_23apr"],
    "JAL8": ["JAL8_flip1_25apr", "JAL8_flip2_29apr", "JAL8_flip4_10may", "JAL8_flip3_7may", "JAL8_14may"],
    "JAL4": ["JAL4_28aug", "JAL4_3rdSept", "JAL4_11thSept", "JAL4_19thSept"],
    "JAL5": ["JAL5_8thSept", "JAL5_21stSept"],
}

# The experiment objects index must match the session name index for the code to work
experiments_objects = [
    JAL6_flip3_18mar,
    JAL6_flip4_21mar,
    JAL6_flip5_25mar,
    JAL6_28mar,
    JAL3_25aug,
    JAL3_1sept,
    JAL3_4sept,
    JAL3_7sept,
    JAL005_8thSept,
    JAL005_21stSept,
    JAL7_sesh8_9apr,
    JAL7_sesh9_16apr,
    JAL7_flip5_22mar,
    JAL7_flip2_12mar,
    JAL7_23apr,
    JAL8_flip1_25apr,
    JAL8_flip2_29apr,
    JAL8_flip4_10may,
    JAL8_14may,
    JAL8_flip3_7may,
    JAL4_3rdSept,
    JAL4_19thSept,
    JAL4_28aug,
    JAL4_11thSept,
]

# The session name indexes must match the experiment object indexes
session_names = [
    "JAL6_flip3_18mar",
    "JAL6_flip4_21mar",
    "JAL6_flip5_25mar",
    "JAL6_28mar",
    "JAL3_25aug",
    "JAL3_1sept",
    "JAL3_4sept",
    "JAL3_7sept",
    "JAL5_8thSept",
    "JAL5_21stSept",
    "JAL7_sesh8_9apr",
    "JAL7_sesh9_16apr",
    "JAL7_flip5_22mar",
    "JAL7_flip2_12mar",
    "JAL7_23apr",
    "JAL8_flip1_25apr",
    "JAL8_flip2_29apr",
    "JAL8_flip4_10may",
    "JAL8_14may",
    "JAL8_flip3_7may",
    "JAL4_3rdSept",
    "JAL4_19thSept",
    "JAL4_28aug",
    "JAL4_11thSept",
]

In [4]:
def classify_hdir(session: object, cluster_type: str = "good") -> list:
    """Label cells as head direction based on a set of criteria

    Returns:
    -- cell ids (list) that are head direction cells"""
    path = extract_rayleigh_path(session, cluster_type, condition="shelter_only", file_name="hdir_Rayleigh.arrow")
    data = load_rayleigh_data(path)
    angles = extract_compartment_values(data, "Rayleigh_theta")
    magnitude = extract_compartment_values(data, "Rayleigh")
    sig = extract_compartment_values(data, "Rayleigh_sig")
    head_direction_cells = []
    for row, cell_id in enumerate(data["clusterID"]):
        simlar_angles = angle_similarity(angles[row][0], angles[row][1])
        above_threshold = rayleigh_threshold(magnitude[row][0], magnitude[row][1], RAYLEIGH_THRESHOLD = 0.4)
        sig_bool = check_both_compartments_significant(sig[row])
        if simlar_angles > SIMILAR_ANGLE_THRESHOLD and above_threshold and sig_bool:
            head_direction_cells.append(cell_id)
    logger.info(f"Found {len(head_direction_cells)} head direction cells")
    logger.info(f"Head direction cells: {head_direction_cells}")
    return head_direction_cells

def classify_hsa(session: object, hdir_cells: list) -> list:
    """Label cells as head shelter based on a set of criteria

    Inputs:
    -- hdir_cells: list of cell ids that are head direction cells in
    order to filter them out of the head shelter cells

    Returns:
    -- cell ids (list) that are head shelter direction cells"""

    path = extract_rayleigh_path(session, cluster_type = "good", condition="shelter_only", file_name="hsa_Rayleigh.arrow")
    data = load_rayleigh_data(path)

    angles = extract_compartment_values(data, "Rayleigh_theta")
    magnitude = extract_compartment_values(data, "Rayleigh")
    sig = extract_compartment_values(data, "Rayleigh_sig")

    head_shelter_cells = []
    for row, cell_id in enumerate(data["clusterID"]):
        simlar_angles = angle_similarity(angles[row][0], angles[row][1])
        above_threshold = rayleigh_threshold(magnitude[row][0], magnitude[row][1], RAYLEIGH_THRESHOLD = 0.2)
        sig_bool = check_both_compartments_significant(sig[row])
        if simlar_angles > SIMILAR_ANGLE_THRESHOLD and above_threshold and sig_bool:
            head_shelter_cells.append(cell_id)

    hsa_cells = [cell for cell in head_shelter_cells if cell not in hdir_cells]

    return hsa_cells


In [8]:
from collections import defaultdict
head_direction_cells = defaultdict(list)

for experiment, session_name in zip(experiments_objects, session_names):
    logger.info(f"Loading data for session: {session_name}")
    loaded_session = get_experiment(experiment)
    hdir_cells = classify_hdir(loaded_session)
    hsa_cells = classify_hsa(loaded_session, hdir_cells)
    head_direction_cells[session_name] = hdir_cells

2025-01-30 14:31:17.892 | INFO     | __main__:<module>:5 - Loading data for session: JAL6_flip3_18mar
2025-01-30 14:31:17.993 | INFO     | __main__:classify_hdir:18 - Found 8 head direction cells
2025-01-30 14:31:17.994 | INFO     | __main__:classify_hdir:19 - Head direction cells: [140.0, 207.0, 212.0, 266.0, 435.0, 474.0, 517.0, 546.0]
2025-01-30 14:31:24.632 | INFO     | __main__:<module>:5 - Loading data for session: JAL6_flip4_21mar
2025-01-30 14:31:24.664 | INFO     | __main__:classify_hdir:18 - Found 13 head direction cells
2025-01-30 14:31:24.665 | INFO     | __main__:classify_hdir:19 - Head direction cells: [150.0, 152.0, 187.0, 213.0, 236.0, 406.0, 444.0, 466.0, 467.0, 470.0, 578.0, 579.0, 610.0]
2025-01-30 14:31:24.698 | INFO     | __main__:<module>:5 - Loading data for session: JAL6_flip5_25mar
2025-01-30 14:31:24.731 | INFO     | __main__:classify_hdir:18 - Found 4 head direction cells
2025-01-30 14:31:24.731 | INFO     | __main__:classify_hdir:19 - Head direction cells: [

In [8]:
head_direction_cells
from pathlib import Path
dir = Path("Z:\Jasmine_Laurence\single_trial_overview\decoding_spatial_efficiency")

# Save the head direction cells
import pickle
with open(dir / "head_direction_cells.pkl", "wb") as f:
    pickle.dump(head_direction_cells, f)
